# AI Comic Translator para Google Colab

Notebook listo para traducir páginas de cómics del inglés al español, con OCR, traducción, limpieza de texto original, renderizado y exportación en ZIP.

Flujo principal:
1. Subes imágenes o un ZIP.
2. El notebook detecta texto con OCR.
3. Traduce automáticamente.
4. Borra el texto original con inpainting.
5. Reescribe la traducción dentro de los globos.
6. Guarda y descarga un ZIP con el resultado.

In [ ]:
# 1) Instalar dependencias necesarias
import sys
import subprocess
import importlib.util


def ensure_package(package_name, module_name=None):
    module_name = module_name or package_name.replace('-', '_')
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])


packages = {
    'easyocr': 'easyocr',
    'opencv-python': 'cv2',
    'pillow': 'PIL',
    'numpy': 'numpy',
    'deep-translator': 'deep_translator',
    'matplotlib': 'matplotlib',
    'requests': 'requests',
    'beautifulsoup4': 'bs4',
    'pytesseract': 'pytesseract',
    'tqdm': 'tqdm',
    'rapidfuzz': 'rapidfuzz',
    'wordfreq': 'wordfreq',
    'google-generativeai': 'google.generativeai',
}

for package_name, module_name in packages.items():
    ensure_package(package_name, module_name)

# PaddleOCR se mantiene opcional para no penalizar tiempo de arranque.

try:
    subprocess.run(['apt-get', 'update'], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(['apt-get', 'install', '-y', 'tesseract-ocr'], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
except Exception as exc:
    print('Instalación opcional de Tesseract omitida:', exc)

print('Dependencias listas.')

In [ ]:
# 2) Imports y configuración general
import os
import re
import math
import json
import shutil
import zipfile
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm
import requests
from bs4 import BeautifulSoup
from deep_translator import GoogleTranslator
from rapidfuzz import process, fuzz
from wordfreq import top_n_list
import easyocr
import google.generativeai as genai

try:
    import pytesseract
except Exception:
    pytesseract = None

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

OUTPUT_ROOT = Path('/content/comic_translation_output') if IN_COLAB else Path('comic_translation_output')
INPUT_DIR = OUTPUT_ROOT / 'input'
OUTPUT_DIR = OUTPUT_ROOT / 'translated'
DEBUG_DIR = OUTPUT_ROOT / 'debug'

for folder in [INPUT_DIR, OUTPUT_DIR, DEBUG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'source_lang': 'en',
    'target_lang': 'es',
    'keep_original_subtitle': False,
    'use_gpu': False,
    'ocr_engine': 'easyocr',  # easyocr | tesseract | paddle
    'ocr_min_confidence': 0.40,
    'retry_low_confidence': True,
    'low_confidence_threshold': 0.60,
    'ocr_use_paragraph_pass': True,
    'ocr_include_gray_pass': True,
    'ocr_scales': [1.0, 1.35],
    'ocr_dedupe_iou': 0.52,
    'ocr_merge_gap_ratio': 0.60,
    'ocr_merge_max_gap_px': 26,
    'ocr_merge_use_near_block': False,
    'ocr_merge_max_union_growth': 1.45,
    'refine_ocr_boxes': True,
    'box_refine_expand_px': 4,
    'box_refine_component_min_area_ratio': 0.0015,
    'enable_ocr_correction': True,
    'correction_min_word_len': 4,
    'correction_similarity_threshold': 86,
    'use_ai_context_correction': False,
    'gemini_api_key': '',
    'gemini_model': 'gemini-1.5-flash',
    'preprocess_use_clahe': True,
    'preprocess_adaptive_block_size': 31,
    'preprocess_adaptive_c': 10,
    'preprocess_blur_kernel': 3,
    'padding': 8,
    'inpaint_radius': 3,
    'max_font_size': 46,
    'min_font_size': 12,
    'uniform_font_size': True,
    'fixed_font_size': 0,  # 0 = automatico, >0 fuerza tamano uniforme
    'auto_calibrate_page': True,
    'auto_font_height_ratio': 0.50,
    'subtitle_ratio': 0.55,
    'translation_chunk_size': 280,
    'show_previews': True,
    'preview_pages': 1,
    'manual_box_mode': 'ask',  # ask | always | never
    'manual_selection_method': 'auto',  # auto | text | click
    'manual_click_timeout': 60,
    'manual_per_page_prompt': True,
    'output_zip_name': 'comic_translated_pages.zip',
    'use_url_input': False,
    'input_url': '',
}

SUPPORTED_TARGET_LANGS = {
    'es': 'Spanish',
    'en': 'English',
    'fr': 'French',
    'pt': 'Portuguese',
    'it': 'Italian',
    'de': 'German',
}

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
TRANSLATION_CACHE = {}
TRANSLATOR_CACHE = {}
OCR_READER = None
PADDLE_OCR_READER = None
FONT_PATH_CACHE = None
WORDLIST_CACHE = {}
AI_MODEL_CACHE = {}
MANUAL_PAGE_DECISIONS = {}

print('Entorno listo.')

In [ ]:
# 3) Utilidades de entrada: subir ZIP o imágenes, o descargar desde URL
def clear_folder(folder_path):
    folder_path = Path(folder_path)
    if folder_path.exists():
        for child in folder_path.iterdir():
            if child.is_file() or child.is_symlink():
                child.unlink()
            elif child.is_dir():
                shutil.rmtree(child)
    folder_path.mkdir(parents=True, exist_ok=True)

def is_image_file(path):
    return Path(path).suffix.lower() in IMAGE_EXTENSIONS

def guess_extension(content_type, fallback_url=''):
    content_type = (content_type or '').lower()
    fallback_url = fallback_url.lower()
    if 'png' in content_type or fallback_url.endswith('.png'):
        return '.png'
    if 'webp' in content_type or fallback_url.endswith('.webp'):
        return '.webp'
    if 'bmp' in content_type or fallback_url.endswith('.bmp'):
        return '.bmp'
    return '.jpg'

def extract_zip_file(zip_path, destination):
    extracted = []
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(destination)
    for file_path in Path(destination).rglob('*'):
        if file_path.is_file() and is_image_file(file_path):
            extracted.append(file_path)
    return extracted

def collect_image_files(root_folder):
    files_found = []
    for file_path in sorted(Path(root_folder).rglob('*')):
        if file_path.is_file() and is_image_file(file_path):
            files_found.append(file_path)
    unique_files = []
    seen = set()
    for file_path in files_found:
        resolved = str(file_path.resolve())
        if resolved not in seen:
            unique_files.append(file_path)
            seen.add(resolved)
    return unique_files

def download_images_from_url(url, destination):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, timeout=30, headers=headers)
    response.raise_for_status()

    content_type = response.headers.get('content-type', '').lower()
    saved_paths = []

    if 'image' in content_type or re.search(r'\.(jpg|jpeg|png|webp|bmp)(\?|$)', url.lower()):
        extension = guess_extension(content_type, url)
        output_path = destination / f'downloaded_001{extension}'
        output_path.write_bytes(response.content)
        return [output_path]

    soup = BeautifulSoup(response.text, 'html.parser')
    candidate_urls = []
    for image_tag in soup.find_all('img'):
        source = image_tag.get('data-src') or image_tag.get('src') or image_tag.get('data-lazy-src')
        if not source:
            continue
        full_url = requests.compat.urljoin(url, source)
        if full_url not in candidate_urls:
            candidate_urls.append(full_url)

    if not candidate_urls:
        for link_tag in soup.find_all('a', href=True):
            href = requests.compat.urljoin(url, link_tag['href'])
            if re.search(r'\.(jpg|jpeg|png|webp|bmp)(\?|$)', href.lower()):
                candidate_urls.append(href)

    if not candidate_urls:
        raise ValueError('No se encontraron imágenes en la URL proporcionada.')

    for index, image_url in enumerate(candidate_urls, start=1):
        try:
            image_response = requests.get(image_url, timeout=30, headers=headers)
            image_response.raise_for_status()
            extension = guess_extension(image_response.headers.get('content-type', ''), image_url)
            output_path = destination / f'url_{index:03d}{extension}'
            output_path.write_bytes(image_response.content)
            saved_paths.append(output_path)
        except Exception as exc:
            print(f'No se pudo descargar {image_url}: {exc}')

    return saved_paths

def prepare_input_images():
    clear_folder(INPUT_DIR)

    if CONFIG['use_url_input'] and CONFIG['input_url'].strip():
        print('Descargando imágenes desde URL...')
        downloaded = download_images_from_url(CONFIG['input_url'].strip(), INPUT_DIR)
        return sorted(downloaded)

    if IN_COLAB:
        if files is None:
            raise RuntimeError('No se pudo acceder a google.colab.files.')
        print('Sube imágenes JPG/PNG o un ZIP con páginas del cómic.')
        uploaded = files.upload()
        if not uploaded:
            raise ValueError('No se subió ningún archivo.')

        for filename, file_bytes in uploaded.items():
            output_path = INPUT_DIR / filename
            output_path.write_bytes(file_bytes)
            if output_path.suffix.lower() == '.zip':
                extract_zip_file(output_path, INPUT_DIR)

        return collect_image_files(INPUT_DIR)

    print('Modo local detectado. Se usarán las imágenes dentro de la carpeta input.')
    return collect_image_files(INPUT_DIR)

In [ ]:
# 4) OCR robusto, corrección de contexto, traducción y renderizado

def clean_text(text):
    text = '' if text is None else str(text)
    text = re.sub(r'\s+', ' ', text).strip()
    replacements = {
        ' .': '.',
        ' ,': ',',
        ' !': '!',
        ' ?': '?',
        ' ;': ';',
        ' :': ':',
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text.strip()


def preprocess_image(image_bgr):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

    if CONFIG['preprocess_use_clahe']:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(gray)
    else:
        enhanced = gray

    blur_k = int(CONFIG['preprocess_blur_kernel'])
    blur_k = blur_k if blur_k % 2 == 1 else blur_k + 1
    denoised = cv2.medianBlur(enhanced, blur_k)

    block_size = int(CONFIG['preprocess_adaptive_block_size'])
    block_size = block_size if block_size % 2 == 1 else block_size + 1
    block_size = max(11, block_size)

    adaptive = cv2.adaptiveThreshold(
        denoised,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        block_size,
        int(CONFIG['preprocess_adaptive_c']),
    )

    edges = cv2.Canny(denoised, 50, 150)

    return {
        'gray': gray,
        'enhanced': enhanced,
        'denoised': denoised,
        'adaptive': adaptive,
        'edges': edges,
    }


def get_reader():
    global OCR_READER
    if OCR_READER is None:
        use_gpu = False
        try:
            import torch
            use_gpu = bool(CONFIG['use_gpu']) and torch.cuda.is_available()
        except Exception:
            use_gpu = False
        OCR_READER = easyocr.Reader(['en'], gpu=use_gpu, verbose=False)
    return OCR_READER


def get_paddle_reader():
    global PADDLE_OCR_READER
    if PADDLE_OCR_READER is not None:
        return PADDLE_OCR_READER
    try:
        from paddleocr import PaddleOCR
        PADDLE_OCR_READER = PaddleOCR(use_angle_cls=True, lang='en', use_gpu=bool(CONFIG['use_gpu']))
        return PADDLE_OCR_READER
    except Exception as exc:
        print('PaddleOCR no disponible, se usará EasyOCR:', exc)
        return None


def get_ai_model():
    if not CONFIG['use_ai_context_correction']:
        return None

    api_key = str(CONFIG.get('gemini_api_key', '')).strip()
    if not api_key:
        return None

    model_name = str(CONFIG.get('gemini_model', 'gemini-1.5-flash')).strip()
    if model_name in AI_MODEL_CACHE:
        return AI_MODEL_CACHE[model_name]

    try:
        genai.configure(api_key=api_key)
        model = genai.GenerativeModel(model_name)
        AI_MODEL_CACHE[model_name] = model
        return model
    except Exception as exc:
        print('No se pudo inicializar Gemini para corrección contextual:', exc)
        return None


def bbox_to_rect(bbox):
    points = np.array(bbox, dtype=np.float32)
    xs = points[:, 0]
    ys = points[:, 1]
    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())


def rect_to_bbox(rect):
    x1, y1, x2, y2 = rect
    return np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]], dtype=np.int32)


def expand_rect(rect, padding, width, height):
    x1, y1, x2, y2 = rect
    return (
        max(0, x1 - padding),
        max(0, y1 - padding),
        min(width - 1, x2 + padding),
        min(height - 1, y2 + padding),
    )


def parse_easyocr_entry(entry):
    if isinstance(entry, (list, tuple)):
        if len(entry) >= 3:
            return entry[0], entry[1], float(entry[2])
        if len(entry) == 2:
            return entry[0], entry[1], 1.0
    return None, '', 0.0


def detect_with_tesseract(image_bgr):
    if pytesseract is None:
        return []

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    data = pytesseract.image_to_data(image_rgb, lang='eng', output_type=pytesseract.Output.DICT)
    grouped = defaultdict(list)

    for index, raw_text in enumerate(data['text']):
        text = clean_text(raw_text)
        try:
            confidence = float(data['conf'][index]) / 100.0
        except Exception:
            confidence = 0.0

        if not text or confidence < float(CONFIG['ocr_min_confidence']):
            continue

        key = (data['block_num'][index], data['par_num'][index], data['line_num'][index])
        x = int(data['left'][index])
        y = int(data['top'][index])
        w = int(data['width'][index])
        h = int(data['height'][index])
        grouped[key].append((x, y, w, h, text, confidence))

    items = []
    for group in grouped.values():
        xs = [entry[0] for entry in group]
        ys = [entry[1] for entry in group]
        xe = [entry[0] + entry[2] for entry in group]
        ye = [entry[1] + entry[3] for entry in group]
        text = clean_text(' '.join(entry[4] for entry in group))
        if not text:
            continue
        items.append({
            'bbox': rect_to_bbox((min(xs), min(ys), max(xe), max(ye))),
            'text': text,
            'confidence': float(np.mean([entry[5] for entry in group])),
            'engine': 'tesseract',
        })

    items.sort(key=lambda item: (bbox_to_rect(item['bbox'])[1], bbox_to_rect(item['bbox'])[0]))
    return items


def detect_with_paddle(image_bgr):
    reader = get_paddle_reader()
    if reader is None:
        return []

    results = reader.ocr(image_bgr, cls=True)
    items = []
    if not results:
        return items

    for row in results:
        for det in row:
            bbox, txt_score = det
            if not txt_score or len(txt_score) < 2:
                continue
            text = clean_text(txt_score[0])
            confidence = float(txt_score[1])
            if not text or confidence < float(CONFIG['ocr_min_confidence']):
                continue
            items.append({
                'bbox': np.array(bbox, dtype=np.int32),
                'text': text,
                'confidence': confidence,
                'engine': 'paddle',
            })

    items.sort(key=lambda item: (bbox_to_rect(item['bbox'])[1], bbox_to_rect(item['bbox'])[0]))
    return items


def detect_with_easyocr(image_input, paragraph=False):
    reader = get_reader()
    results = reader.readtext(
        image_input,
        detail=1,
        paragraph=paragraph,
        min_size=10,
        text_threshold=0.35,
        low_text=0.25,
        decoder='beamsearch',
    )

    items = []
    for entry in results:
        bbox, text, confidence = parse_easyocr_entry(entry)
        text = clean_text(text)
        if bbox is None or not text:
            continue
        if float(confidence) < float(CONFIG['ocr_min_confidence']):
            continue
        items.append({
            'bbox': np.array(bbox, dtype=np.int32),
            'text': text,
            'confidence': float(confidence),
            'engine': 'easyocr',
        })

    items.sort(key=lambda item: (bbox_to_rect(item['bbox'])[1], bbox_to_rect(item['bbox'])[0]))
    return items


def scale_items_back(items, scale):
    scale = float(scale)
    if abs(scale - 1.0) < 1e-6:
        return items

    inv = 1.0 / scale
    scaled_items = []
    for item in items:
        x1, y1, x2, y2 = bbox_to_rect(item['bbox'])
        scaled_items.append({
            **item,
            'bbox': rect_to_bbox((
                int(round(x1 * inv)),
                int(round(y1 * inv)),
                int(round(x2 * inv)),
                int(round(y2 * inv)),
            )),
            'engine': f"{item.get('engine', 'easyocr')}-scaled" if scale != 1.0 else item.get('engine', 'easyocr'),
        })
    return scaled_items


def rect_iou(rect_a, rect_b):
    ax1, ay1, ax2, ay2 = rect_a
    bx1, by1, bx2, by2 = rect_b
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h
    if inter_area <= 0:
        return 0.0
    area_a = max(1, (ax2 - ax1) * (ay2 - ay1))
    area_b = max(1, (bx2 - bx1) * (by2 - by1))
    return inter_area / float(area_a + area_b - inter_area)


def rect_containment(rect_a, rect_b):
    ax1, ay1, ax2, ay2 = rect_a
    bx1, by1, bx2, by2 = rect_b
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h
    area_a = max(1, (ax2 - ax1) * (ay2 - ay1))
    area_b = max(1, (bx2 - bx1) * (by2 - by1))
    return max(inter_area / float(area_a), inter_area / float(area_b))


def merge_text_values(text_a, text_b):
    text_a = clean_text(text_a)
    text_b = clean_text(text_b)
    if not text_a:
        return text_b
    if not text_b:
        return text_a
    low_a = text_a.lower()
    low_b = text_b.lower()
    if low_a in low_b:
        return text_b
    if low_b in low_a:
        return text_a
    return clean_text(f"{text_a} {text_b}")


def should_merge_rects(rect_a, rect_b):
    ax1, ay1, ax2, ay2 = rect_a
    bx1, by1, bx2, by2 = rect_b
    ha = max(1, ay2 - ay1)
    hb = max(1, by2 - by1)
    wa = max(1, ax2 - ax1)
    wb = max(1, bx2 - bx1)
    avg_h = (ha + hb) / 2.0

    max_gap_px = int(CONFIG.get('ocr_merge_max_gap_px', 26))
    max_gap_ratio = float(CONFIG.get('ocr_merge_gap_ratio', 0.60))
    max_gap = max(max_gap_px, int(avg_h * max_gap_ratio))

    horizontal_gap = max(0, max(ax1, bx1) - min(ax2, bx2))
    vertical_gap = max(0, max(ay1, by1) - min(ay2, by2))

    overlap_w = max(0, min(ax2, bx2) - max(ax1, bx1))
    overlap_h = max(0, min(ay2, by2) - max(ay1, by1))
    overlap_area = overlap_w * overlap_h

    same_line = overlap_h >= int(min(ha, hb) * 0.35) and horizontal_gap <= max_gap
    same_column = overlap_w >= int(min(ax2 - ax1, bx2 - bx1) * 0.35) and vertical_gap <= max_gap
    near_block = horizontal_gap <= int(max_gap * 0.8) and vertical_gap <= int(max_gap * 0.8)

    union_w = max(ax2, bx2) - min(ax1, bx1)
    union_h = max(ay2, by2) - min(ay1, by1)
    union_area = max(1, union_w * union_h)
    sum_area = max(1, (wa * ha) + (wb * hb))
    max_union_growth = float(CONFIG.get('ocr_merge_max_union_growth', 1.45))
    if (union_area / float(sum_area)) > max_union_growth:
        return False

    use_near_block = bool(CONFIG.get('ocr_merge_use_near_block', False))
    near_block_ok = use_near_block and near_block and overlap_area > 0
    return same_line or same_column or near_block_ok


def dedupe_and_merge_ocr_items(items):
    if not items:
        return []

    dedupe_iou = float(CONFIG.get('ocr_dedupe_iou', 0.52))
    deduped = []
    for candidate in sorted(items, key=lambda it: float(it.get('confidence', 0.0)), reverse=True):
        crect = bbox_to_rect(candidate['bbox'])
        replaced = False
        skip = False

        for index, chosen in enumerate(deduped):
            rrect = bbox_to_rect(chosen['bbox'])
            overlap = rect_iou(crect, rrect)
            containment = rect_containment(crect, rrect)
            if overlap >= dedupe_iou or containment >= 0.82:
                cand_len = len(clean_text(candidate.get('text', '')))
                chos_len = len(clean_text(chosen.get('text', '')))
                cand_conf = float(candidate.get('confidence', 0.0))
                chos_conf = float(chosen.get('confidence', 0.0))
                if cand_len >= chos_len and cand_conf + 0.02 >= chos_conf:
                    deduped[index] = candidate
                    replaced = True
                else:
                    skip = True
                break

        if skip:
            continue
        if not replaced:
            deduped.append(candidate)

    changed = True
    while changed:
        changed = False
        merged = []
        used = [False] * len(deduped)

        ordered = sorted(range(len(deduped)), key=lambda idx: (bbox_to_rect(deduped[idx]['bbox'])[1], bbox_to_rect(deduped[idx]['bbox'])[0]))

        for pos, idx in enumerate(ordered):
            if used[idx]:
                continue

            base = deduped[idx]
            brect = bbox_to_rect(base['bbox'])
            btext = base.get('text', '')
            bconf = float(base.get('confidence', 0.0))
            used[idx] = True

            for jdx in ordered[pos + 1:]:
                if used[jdx]:
                    continue

                other = deduped[jdx]
                orect = bbox_to_rect(other['bbox'])
                if not should_merge_rects(brect, orect):
                    continue

                x1 = min(brect[0], orect[0])
                y1 = min(brect[1], orect[1])
                x2 = max(brect[2], orect[2])
                y2 = max(brect[3], orect[3])
                brect = (x1, y1, x2, y2)
                btext = merge_text_values(btext, other.get('text', ''))
                bconf = max(bconf, float(other.get('confidence', 0.0)))
                used[jdx] = True
                changed = True

            merged.append({
                **base,
                'bbox': rect_to_bbox(brect),
                'text': clean_text(btext),
                'confidence': bconf,
                'engine': 'easyocr-merged',
            })

        deduped = merged

    deduped.sort(key=lambda item: (bbox_to_rect(item['bbox'])[1], bbox_to_rect(item['bbox'])[0]))
    return deduped


def detect_with_easyocr_multirun(image_bgr, preprocessed):
    variants = [preprocessed['enhanced'], preprocessed['adaptive']]
    if bool(CONFIG.get('ocr_include_gray_pass', True)):
        variants.append(preprocessed['gray'])

    scales = CONFIG.get('ocr_scales', [1.0, 1.35]) or [1.0]
    raw_items = []

    for variant in variants:
        raw_items.extend(detect_with_easyocr(variant, paragraph=False))

        if bool(CONFIG.get('ocr_use_paragraph_pass', True)):
            paragraph_items = detect_with_easyocr(variant, paragraph=True)
            for item in paragraph_items:
                item['engine'] = 'easyocr-paragraph'
            raw_items.extend(paragraph_items)

        for scale in scales:
            scale = float(scale)
            if scale <= 0 or abs(scale - 1.0) < 1e-6:
                continue
            resized = cv2.resize(variant, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
            scaled_items = detect_with_easyocr(resized, paragraph=False)
            raw_items.extend(scale_items_back(scaled_items, scale))

    return dedupe_and_merge_ocr_items(raw_items)


def ask_manual_for_page(page_name, page_index):
    mode = str(CONFIG.get('manual_box_mode', 'ask')).lower().strip()
    if mode == 'always':
        return True
    if mode == 'never':
        return False

    cache_key = f'{page_index}:{page_name}'
    if cache_key in MANUAL_PAGE_DECISIONS:
        return bool(MANUAL_PAGE_DECISIONS[cache_key])

    answer = input(f'Manual para {page_name}? (s/n, default n): ').strip().lower()
    use_manual = answer in ('s', 'si', 'sí', 'y', 'yes')
    MANUAL_PAGE_DECISIONS[cache_key] = use_manual
    return use_manual


def parse_rect_input(rect_str):
    rect_str = rect_str.strip()
    if not rect_str:
        return None
    parts = [p.strip() for p in rect_str.split(',')]
    if len(parts) != 4:
        return None
    try:
        x1, y1, x2, y2 = [int(float(v)) for v in parts]
    except Exception:
        return None
    x1, x2 = min(x1, x2), max(x1, x2)
    y1, y2 = min(y1, y2), max(y1, y2)
    if x2 - x1 < 5 or y2 - y1 < 5:
        return None
    return (x1, y1, x2, y2)


def select_text_regions_manual_text(image_bgr, page_name):
    h, w = image_bgr.shape[:2]
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.imshow(image_rgb)
    ax.set_title(f'{page_name} - Ingrese coordenadas x1,y1,x2,y2')
    ax.axis('off')
    plt.show()

    print(f'Tamano de imagen: {w}x{h}')
    print('Ingresa una region por linea con formato: x1,y1,x2,y2')
    print('Cuando termines, presiona Enter en blanco.')

    manual_rects = []
    while True:
        raw = input('Region: ').strip()
        if not raw:
            break
        rect = parse_rect_input(raw)
        if rect is None:
            print('Formato invalido o region muy pequena. Intenta de nuevo.')
            continue
        x1, y1, x2, y2 = rect
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w - 1, x2), min(h - 1, y2)
        manual_rects.append((x1, y1, x2, y2))
        print(f'Region agregada: {(x1, y1, x2, y2)}')

    return manual_rects


def select_text_regions_manual_click(image_bgr, page_name):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    manual_rects = []
    timeout = int(CONFIG.get('manual_click_timeout', 60))
    print(f'Seleccion manual para {page_name}')
    print('Haz 2 clics por region: esquina superior-izquierda y esquina inferior-derecha.')
    print(f'Si no haces clic en {timeout}s, se cancela esta region para evitar bloqueo.')

    while True:
        fig, ax = plt.subplots(1, 1, figsize=(12, 8))
        ax.imshow(image_rgb)
        ax.set_title('Selecciona 2 puntos para una region de texto')
        ax.axis('off')
        points = plt.ginput(2, timeout=timeout)
        plt.close(fig)

        if len(points) < 2:
            print('No se capturaron 2 clics. Cambiando a entrada por coordenadas...')
            extra = select_text_regions_manual_text(image_bgr, page_name)
            manual_rects.extend(extra)
            break

        x1, y1 = points[0]
        x2, y2 = points[1]
        x1, x2 = int(min(x1, x2)), int(max(x1, x2))
        y1, y2 = int(min(y1, y2)), int(max(y1, y2))

        if x2 - x1 < 5 or y2 - y1 < 5:
            print('Region muy pequena, intenta de nuevo.')
        else:
            manual_rects.append((x1, y1, x2, y2))
            print(f'Region agregada: {(x1, y1, x2, y2)}')

        more = input('Agregar otra region? (s/n, default n): ').strip().lower()
        if more not in ('s', 'si', 'sí', 'y', 'yes'):
            break

    return manual_rects


def select_text_regions_manual(image_bgr, page_name):
    method = str(CONFIG.get('manual_selection_method', 'auto')).lower().strip()
    if method not in ('auto', 'text', 'click'):
        method = 'auto'

    if method == 'text':
        return select_text_regions_manual_text(image_bgr, page_name)

    if method == 'click':
        return select_text_regions_manual_click(image_bgr, page_name)

    # auto: en Colab preferimos texto para evitar bloqueos con ginput.
    if IN_COLAB:
        return select_text_regions_manual_text(image_bgr, page_name)

    return select_text_regions_manual_click(image_bgr, page_name)


def detect_text_in_manual_regions(image_bgr, preprocessed, manual_rects):
    items = []
    h, w = image_bgr.shape[:2]

    for order, rect in enumerate(manual_rects):
        x1, y1, x2, y2 = rect
        x1, y1, x2, y2 = expand_rect((x1, y1, x2, y2), 0, w, h)
        roi_enhanced = preprocessed['enhanced'][y1:y2, x1:x2]
        roi_bgr = image_bgr[y1:y2, x1:x2]

        roi_items = detect_with_easyocr(roi_enhanced, paragraph=True)
        if not roi_items:
            roi_items = detect_with_tesseract(roi_bgr)

        if not roi_items:
            continue

        merged_text = clean_text(' '.join(item['text'] for item in roi_items))
        merged_conf = float(np.mean([item.get('confidence', 0.0) for item in roi_items]))
        items.append({
            'bbox': rect_to_bbox((x1, y1, x2, y2)),
            'text': merged_text,
            'confidence': merged_conf,
            'engine': 'manual-region',
            'manual_order': order,
        })

    items.sort(key=lambda it: it.get('manual_order', 999999))
    return items


def detect_text(image_bgr, preprocessed, manual_rects=None):
    if manual_rects:
        manual_items = detect_text_in_manual_regions(image_bgr, preprocessed, manual_rects)
        if manual_items:
            return manual_items

    engine = str(CONFIG.get('ocr_engine', 'easyocr')).lower().strip()

    if engine == 'tesseract':
        items = detect_with_tesseract(image_bgr)
    elif engine == 'paddle':
        items = detect_with_paddle(image_bgr)
        if not items:
            items = detect_with_easyocr_multirun(image_bgr, preprocessed)
    else:
        items = detect_with_easyocr_multirun(image_bgr, preprocessed)

    if not items:
        items = detect_with_tesseract(image_bgr)

    return items


def retry_low_confidence_regions(image_bgr, preprocessed, items):
    if not CONFIG['retry_low_confidence']:
        return items

    height, width = image_bgr.shape[:2]
    retried_items = []
    reader = get_reader()

    for item in items:
        if float(item.get('confidence', 0.0)) >= float(CONFIG['low_confidence_threshold']):
            retried_items.append(item)
            continue

        x1, y1, x2, y2 = expand_rect(bbox_to_rect(item['bbox']), 10, width, height)
        roi = preprocessed['enhanced'][y1:y2, x1:x2]
        if roi.size == 0:
            retried_items.append(item)
            continue

        roi_results = reader.readtext(
            roi,
            detail=1,
            paragraph=False,
            decoder='beamsearch',
            min_size=8,
            text_threshold=0.30,
            low_text=0.20,
        )

        best = item
        for entry in roi_results:
            rbbox, rtext, rconf = parse_easyocr_entry(entry)
            rtext = clean_text(rtext)
            if not rtext:
                continue
            if float(rconf) > float(best.get('confidence', 0.0)):
                best = {
                    'bbox': item['bbox'],
                    'text': rtext,
                    'confidence': float(rconf),
                    'engine': 'easyocr-retry',
                }

        retried_items.append(best)

    retried_items.sort(key=lambda it: (bbox_to_rect(it['bbox'])[1], bbox_to_rect(it['bbox'])[0]))
    return retried_items


def refine_ocr_boxes(items, preprocessed, image_shape):
    if not items or not bool(CONFIG.get('refine_ocr_boxes', True)):
        return items

    adaptive = preprocessed.get('adaptive') if isinstance(preprocessed, dict) else None
    if adaptive is None:
        return items

    inv = cv2.bitwise_not(adaptive)
    height, width = image_shape[:2]
    expand_px = int(CONFIG.get('box_refine_expand_px', 4))
    area_ratio = float(CONFIG.get('box_refine_component_min_area_ratio', 0.0015))

    refined = []
    for item in items:
        x1, y1, x2, y2 = bbox_to_rect(item['bbox'])
        x1, y1, x2, y2 = expand_rect((x1, y1, x2, y2), expand_px, width, height)

        roi = inv[y1:y2, x1:x2]
        box_w = max(1, x2 - x1)
        box_h = max(1, y2 - y1)
        box_area = max(1, box_w * box_h)

        if roi.size == 0:
            refined.append(item)
            continue

        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(roi, connectivity=8)
        min_comp_area = max(6, int(box_area * area_ratio))

        xs = []
        ys = []
        xes = []
        yes = []

        for label in range(1, num_labels):
            area = int(stats[label, cv2.CC_STAT_AREA])
            if area < min_comp_area:
                continue
            cx = int(stats[label, cv2.CC_STAT_LEFT])
            cy = int(stats[label, cv2.CC_STAT_TOP])
            cw = int(stats[label, cv2.CC_STAT_WIDTH])
            ch = int(stats[label, cv2.CC_STAT_HEIGHT])
            xs.append(cx)
            ys.append(cy)
            xes.append(cx + cw)
            yes.append(cy + ch)

        if not xs:
            refined.append(item)
            continue

        rx1 = max(0, x1 + min(xs) - 2)
        ry1 = max(0, y1 + min(ys) - 2)
        rx2 = min(width - 1, x1 + max(xes) + 2)
        ry2 = min(height - 1, y1 + max(yes) + 2)

        new_area = max(1, (rx2 - rx1) * (ry2 - ry1))
        area_ratio_new = new_area / float(box_area)
        if area_ratio_new < 0.25 or area_ratio_new > 1.20:
            refined.append(item)
            continue

        refined.append({
            **item,
            'bbox': rect_to_bbox((rx1, ry1, rx2, ry2)),
            'engine': f"{item.get('engine', 'easyocr')}-refined"
        })

    refined.sort(key=lambda it: (bbox_to_rect(it['bbox'])[1], bbox_to_rect(it['bbox'])[0]))
    return refined


def estimate_page_font_size(items):
    if not items:
        return None

    heights = []
    widths = []
    char_densities = []

    for item in items:
        x1, y1, x2, y2 = bbox_to_rect(item['bbox'])
        w = max(1, x2 - x1)
        h = max(1, y2 - y1)
        txt = clean_text(item.get('translated', item.get('text', '')))
        txt_len = max(1, len(txt))
        heights.append(h)
        widths.append(w)
        char_densities.append(txt_len / float(w))

    if not heights:
        return None

    median_h = float(np.median(heights))
    median_density = float(np.median(char_densities)) if char_densities else 0.0
    ratio = float(CONFIG.get('auto_font_height_ratio', 0.50))
    suggested = int(round(median_h * ratio))

    if median_density > 0.11:
        suggested -= 2
    elif median_density < 0.06:
        suggested += 1

    suggested = int(np.clip(suggested, CONFIG['min_font_size'], CONFIG['max_font_size']))
    return max(CONFIG['min_font_size'], suggested)


def get_wordlist(size=50000):
    cache_key = int(size)
    if cache_key in WORDLIST_CACHE:
        return WORDLIST_CACHE[cache_key]
    try:
        words = set(top_n_list('en', cache_key))
    except Exception:
        words = set()
    WORDLIST_CACHE[cache_key] = words
    return words


def _apply_case_pattern(source_token, candidate_token):
    if source_token.isupper():
        return candidate_token.upper()
    if source_token.islower():
        return candidate_token.lower()
    if source_token[:1].isupper() and source_token[1:].islower():
        return candidate_token.capitalize()
    return candidate_token


def _generate_ocr_variants(token, max_variants=80):
    confusion_map = {
        '0': ['o', 'O'],
        '1': ['i', 'l', 'I', 'L', 'u', 'U'],
        '5': ['s', 'S'],
        '8': ['b', 'B'],
        '6': ['g', 'G'],
        '2': ['z', 'Z'],
        '|': ['I', 'L', 'l'],
        '!': ['I', 'L', 'l'],
        'l': ['i', 'u'],
        'L': ['I', 'U'],
        'I': ['l', 'L', 'U'],
    }

    variants = {token}
    current = {token}

    for _ in range(2):
        next_level = set()
        for word in current:
            for idx, char in enumerate(word):
                replacements = confusion_map.get(char, [])
                for repl in replacements:
                    candidate = f"{word[:idx]}{repl}{word[idx + 1:]}"
                    if candidate not in variants:
                        next_level.add(candidate)
                        variants.add(candidate)
                        if len(variants) >= max_variants:
                            return variants
        if not next_level:
            break
        current = next_level

    return variants


def repair_ocr_token(token, dictionary):
    if not token:
        return token

    base_low = token.lower()
    if base_low in dictionary:
        return token

    variants = _generate_ocr_variants(token)
    best_word = ''
    best_score = -1.0

    for variant in variants:
        low_variant = variant.lower()
        if low_variant in dictionary:
            return _apply_case_pattern(token, low_variant)

        candidate = process.extractOne(
            low_variant,
            dictionary,
            scorer=fuzz.ratio,
            score_cutoff=82,
        )
        if not candidate:
            continue

        word, score, _ = candidate
        penalty = abs(len(word) - len(token)) * 2.0
        adjusted = float(score) - penalty
        if adjusted > best_score:
            best_score = adjusted
            best_word = word

    if not best_word:
        return token

    return _apply_case_pattern(token, best_word)


def correct_text(text):
    text = clean_text(text)
    if not CONFIG['enable_ocr_correction'] or not text:
        return text

    dictionary = get_wordlist()
    if not dictionary:
        return text

    tokens = re.findall(r"[A-Za-z0-9'|!]+|[^A-Za-z0-9'|!]+", text)
    corrected_tokens = []

    for token in tokens:
        if not re.match(r"[A-Za-z0-9'|!]+$", token):
            corrected_tokens.append(token)
            continue

        low = token.lower()
        if low in dictionary:
            corrected_tokens.append(token)
            continue

        repaired = repair_ocr_token(token, dictionary)
        repaired_low = repaired.lower()
        if repaired_low in dictionary:
            corrected_tokens.append(repaired)
            continue

        if len(repaired_low) < int(CONFIG['correction_min_word_len']):
            corrected_tokens.append(repaired)
            continue

        candidate = process.extractOne(
            repaired_low,
            dictionary,
            scorer=fuzz.ratio,
            score_cutoff=float(CONFIG['correction_similarity_threshold']),
        )

        if not candidate:
            corrected_tokens.append(repaired)
            continue

        best_word = candidate[0]
        if repaired.isupper():
            best_word = best_word.upper()
        elif repaired[0].isupper():
            best_word = best_word.capitalize()

        corrected_tokens.append(best_word)

    return ''.join(corrected_tokens)


def local_context_reflow(text):
    text = clean_text(text)
    if not text:
        return ''
    text = text.replace(' ,', ',').replace(' .', '.')
    text = re.sub(r'\s*\n\s*', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if text and text[-1] not in '.!?"\'':
        text += '.'
    return text


def ai_context_fix(text, context_hint=''):
    model = get_ai_model()
    if model is None:
        return ''

    prompt = (
        'Corrige OCR en ingles y reorganiza el dialogo para que suene natural. '
        'No inventes informacion. Devuelve solo una frase corregida en ingles.\n\n'
        f'Contexto cercano: {context_hint}\n'
        f'Texto OCR: {text}'
    )

    try:
        response = model.generate_content(prompt)
        candidate = '' if response is None else getattr(response, 'text', '')
        return clean_text(candidate)
    except Exception as exc:
        print('Fallo correccion con IA:', exc)
        return ''


def correct_context(text, context_hint=''):
    base = local_context_reflow(text)
    base = correct_text(base)

    if CONFIG['use_ai_context_correction']:
        ai_fixed = ai_context_fix(base, context_hint=context_hint)
        if ai_fixed:
            return ai_fixed

    return base


def split_translation_chunks(text, chunk_size=280):
    text = clean_text(text)
    if len(text) <= chunk_size:
        return [text]

    chunks = []
    current = ''
    for sentence in re.split(r'(?<=[.!?])\s+', text):
        sentence = sentence.strip()
        if not sentence:
            continue
        if len(current) + len(sentence) + 1 <= chunk_size:
            current = sentence if not current else f'{current} {sentence}'
        else:
            if current:
                chunks.append(current)
            current = sentence
    if current:
        chunks.append(current)
    return chunks if chunks else [text]


def get_translator(target_lang):
    source_lang = str(CONFIG.get('source_lang', 'en') or 'en').strip().lower()
    cache_key = (source_lang, target_lang)
    if cache_key in TRANSLATOR_CACHE:
        return TRANSLATOR_CACHE[cache_key]

    try:
        translator = GoogleTranslator(source=source_lang, target=target_lang)
    except Exception:
        translator = GoogleTranslator(source='auto', target=target_lang)

    TRANSLATOR_CACHE[cache_key] = translator
    return translator


def translate_text(text, target_lang):
    text = clean_text(text)
    if not text:
        return ''

    cache_key = (text, target_lang)
    if cache_key in TRANSLATION_CACHE:
        return TRANSLATION_CACHE[cache_key]

    chunks = split_translation_chunks(text, chunk_size=int(CONFIG['translation_chunk_size']))
    translated_chunks = []

    for chunk in chunks:
        if not chunk:
            continue
        try:
            translator = get_translator(target_lang)
            translated = translator.translate(chunk)
            translated = clean_text(translated)
        except Exception as exc:
            print(f'Fallo traducción de chunk: {exc}')
            try:
                translated = clean_text(GoogleTranslator(source='auto', target=target_lang).translate(chunk))
            except Exception:
                translated = chunk
        translated_chunks.append(translated if translated else chunk)

    final_text = clean_text(' '.join(translated_chunks)) or text
    TRANSLATION_CACHE[cache_key] = final_text
    return final_text


def find_font_path():
    global FONT_PATH_CACHE
    if FONT_PATH_CACHE is not None:
        return FONT_PATH_CACHE

    candidates = []
    try:
        from matplotlib import font_manager
        for font_path in font_manager.findSystemFonts(fontpaths=None, fontext='ttf'):
            lower = font_path.lower()
            score = 0
            if 'comic' in lower:
                score += 10
            if 'rounded' in lower:
                score += 5
            if 'bold' in lower:
                score += 3
            if 'dejavu' in lower or 'liberation' in lower or 'noto' in lower:
                score += 1
            candidates.append((score, font_path))
    except Exception:
        candidates = []

    candidates.sort(reverse=True, key=lambda item: item[0])
    FONT_PATH_CACHE = candidates[0][1] if candidates else None
    return FONT_PATH_CACHE


def measure_text(draw, text, font):
    sample = text if text else 'Ag'
    bbox = draw.textbbox((0, 0), sample, font=font)
    return bbox[2] - bbox[0], bbox[3] - bbox[1]


def split_long_word(draw, word, font, max_width):
    if measure_text(draw, word, font)[0] <= max_width:
        return [word]

    parts = []
    current = ''
    for char in word:
        candidate = current + char
        if not current or measure_text(draw, candidate, font)[0] <= max_width:
            current = candidate
        else:
            parts.append(current)
            current = char
    if current:
        parts.append(current)
    return parts


def wrap_text_to_width(draw, text, font, max_width):
    lines = []
    for paragraph in str(text).split('\n'):
        words = paragraph.split()
        if not words:
            lines.append('')
            continue

        current_line = ''
        for word in words:
            for segment in split_long_word(draw, word, font, max_width):
                candidate = segment if not current_line else current_line + ' ' + segment
                if not current_line or measure_text(draw, candidate, font)[0] <= max_width:
                    current_line = candidate
                else:
                    lines.append(current_line)
                    current_line = segment
        if current_line:
            lines.append(current_line)

def render_translation(image_bgr, items, font_path=None, forced_main_size=None):


def measure_block(draw, lines, font, line_spacing=4):
    total_width = 0
    total_height = 0
    fixed_main_size = int(forced_main_size) if forced_main_size is not None else int(CONFIG.get('fixed_font_size', 0) or 0)
        line_width, line_height = measure_text(draw, line, font)
        total_width = max(total_width, line_width)
        total_height += line_height
        if index < len(lines) - 1:
            total_height += line_spacing
    return total_width, total_height


def choose_font_layout(draw, translated_text, original_text, box_width, box_height, font_path, forced_main_size=None):
    translated_text = clean_text(translated_text)
    original_text = clean_text(original_text)
    subtitle_text = ''
    if CONFIG['keep_original_subtitle'] and original_text and original_text != translated_text:
        subtitle_text = original_text

    padded_width = max(20, box_width - 6)
    padded_height = max(20, box_height - 6)

    if forced_main_size is not None:
        fixed_size = int(np.clip(forced_main_size, CONFIG['min_font_size'], CONFIG['max_font_size']))
        main_font = ImageFont.truetype(font_path, fixed_size) if font_path else ImageFont.load_default()
        main_lines = wrap_text_to_width(draw, translated_text, main_font, padded_width)
        subtitle_lines = []
        subtitle_font = None
        if subtitle_text:
            subtitle_size = max(CONFIG['min_font_size'], int(fixed_size * CONFIG['subtitle_ratio']))
            subtitle_font = ImageFont.truetype(font_path, subtitle_size) if font_path else ImageFont.load_default()
            subtitle_lines = wrap_text_to_width(draw, subtitle_text, subtitle_font, padded_width)
        return {
            'main_font': main_font,
            'main_lines': main_lines,
            'subtitle_font': subtitle_font,
            'subtitle_lines': subtitle_lines,
            'subtitle_text': subtitle_text,
        }

    for main_size in range(CONFIG['max_font_size'], CONFIG['min_font_size'] - 1, -1):
        main_font = ImageFont.truetype(font_path, main_size) if font_path else ImageFont.load_default()
        main_lines = wrap_text_to_width(draw, translated_text, main_font, padded_width)
        main_width, main_height = measure_block(draw, main_lines, main_font, line_spacing=max(2, main_size // 7))
        total_height = main_height
        subtitle_lines = []
        subtitle_font = None

        if subtitle_text:
            subtitle_size = max(CONFIG['min_font_size'], int(main_size * CONFIG['subtitle_ratio']))
            subtitle_font = ImageFont.truetype(font_path, subtitle_size) if font_path else ImageFont.load_default()
            subtitle_lines = wrap_text_to_width(draw, subtitle_text, subtitle_font, padded_width)
            subtitle_width, subtitle_height = measure_block(draw, subtitle_lines, subtitle_font, line_spacing=max(1, subtitle_size // 8))
            total_height += max(2, main_size // 8) + subtitle_height
            main_width = max(main_width, subtitle_width)

        if main_width <= padded_width and total_height <= padded_height:
            return {
                'main_font': main_font,
                'main_lines': main_lines,
                'subtitle_font': subtitle_font,
                'subtitle_lines': subtitle_lines,
                'subtitle_text': subtitle_text,
            }

    main_font = ImageFont.truetype(font_path, CONFIG['min_font_size']) if font_path else ImageFont.load_default()
    subtitle_font = ImageFont.truetype(font_path, max(CONFIG['min_font_size'] - 2, 8)) if (font_path and subtitle_text) else (ImageFont.load_default() if subtitle_text else None)
    main_lines = wrap_text_to_width(draw, translated_text, main_font, padded_width)
    subtitle_lines = wrap_text_to_width(draw, subtitle_text, subtitle_font, padded_width) if subtitle_text else []
    return {
        'main_font': main_font,
        'main_lines': main_lines,
        'subtitle_font': subtitle_font,
        'subtitle_lines': subtitle_lines,
        'subtitle_text': subtitle_text,
    }


def draw_centered_multiline(draw, box, lines, font, fill=(0, 0, 0), line_spacing=4):
    x1, y1, x2, y2 = box
    box_width = x2 - x1
    box_height = y2 - y1
    total_width, total_height = measure_block(draw, lines, font, line_spacing=line_spacing)
    start_y = y1 + max(0, (box_height - total_height) / 2)
    current_y = start_y

    for index, line in enumerate(lines):
        line_width, line_height = measure_text(draw, line, font)
        current_x = x1 + max(0, (box_width - line_width) / 2)
        stroke_width = max(1, getattr(font, 'size', CONFIG['min_font_size']) // 18)
        draw.text((current_x, current_y), line, font=font, fill=fill, stroke_width=stroke_width, stroke_fill='white')
        current_y += line_height
        if index < len(lines) - 1:
            current_y += line_spacing


def inpaint_text_regions(image_bgr, items):
    mask = np.zeros(image_bgr.shape[:2], dtype=np.uint8)
    height, width = mask.shape
    for item in items:
        x1, y1, x2, y2 = expand_rect(bbox_to_rect(item['bbox']), CONFIG['padding'], width, height)
        cv2.rectangle(mask, (x1, y1), (x2, y2), 255, -1)
    return cv2.inpaint(image_bgr, mask, CONFIG['inpaint_radius'], cv2.INPAINT_TELEA), mask


def render_translation(image_bgr, items, font_path=None):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(image_rgb)
    draw = ImageDraw.Draw(pil_image)

    uniform_main_size = None
    fixed_main_size = int(CONFIG.get('fixed_font_size', 0) or 0)
    if fixed_main_size > 0:
        uniform_main_size = int(np.clip(fixed_main_size, CONFIG['min_font_size'], CONFIG['max_font_size']))
    elif CONFIG.get('uniform_font_size', True):
        candidate_sizes = []
        for item in items:
            x1, y1, x2, y2 = bbox_to_rect(item['bbox'])
            layout = choose_font_layout(draw, item.get('translated', item['text']), item['text'], x2 - x1, y2 - y1, font_path)
            candidate_sizes.append(getattr(layout['main_font'], 'size', CONFIG['min_font_size']))
        if candidate_sizes:
            uniform_main_size = max(CONFIG['min_font_size'], min(candidate_sizes))

    for item in items:
        x1, y1, x2, y2 = bbox_to_rect(item['bbox'])
        layout = choose_font_layout(
            draw,
            item.get('translated', item['text']),
            item['text'],
            x2 - x1,
            y2 - y1,
            font_path,
            forced_main_size=uniform_main_size,
        )
        draw_centered_multiline(
            draw,
            (x1, y1, x2, y2),
            layout['main_lines'],
            layout['main_font'],
            fill=(0, 0, 0),
            line_spacing=max(2, getattr(layout['main_font'], 'size', CONFIG['min_font_size']) // 7),
        )

    return cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)


def annotate_ocr_preview(image_bgr, items):
    preview = cv2.cvtColor(image_bgr.copy(), cv2.COLOR_BGR2RGB)
    for index, item in enumerate(items, start=1):
        bbox = np.array(item['bbox'], dtype=np.int32)
        cv2.polylines(preview, [bbox], isClosed=True, color=(255, 0, 0), thickness=2)
        x1, y1, x2, y2 = bbox_to_rect(bbox)
        label = f"{index}: {item['text'][:32]} ({item.get('confidence', 0):.2f})"
        cv2.putText(preview, label, (x1, max(14, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 0, 0), 1, cv2.LINE_AA)
    return preview

In [ ]:
# 5) Visualizaciones y pipeline principal

def show_step_by_step(original_bgr, preprocessed, mask, ocr_items, final_bgr, title=''):
    original_rgb = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)
    preproc_rgb = cv2.cvtColor(preprocessed['enhanced'], cv2.COLOR_GRAY2RGB)
    boxes_rgb = annotate_ocr_preview(original_bgr, ocr_items)
    final_rgb = cv2.cvtColor(final_bgr, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    axes[0, 0].imshow(original_rgb)
    axes[0, 0].set_title('Original')
    axes[0, 0].axis('off')

    axes[0, 1].imshow(preproc_rgb)
    axes[0, 1].set_title('Preprocesada (CLAHE + denoise)')
    axes[0, 1].axis('off')

    axes[0, 2].imshow(preprocessed['adaptive'], cmap='gray')
    axes[0, 2].set_title('Binarización adaptativa')
    axes[0, 2].axis('off')

    axes[1, 0].imshow(boxes_rgb)
    axes[1, 0].set_title('Bounding boxes OCR')
    axes[1, 0].axis('off')

    axes[1, 1].imshow(mask, cmap='gray')
    axes[1, 1].set_title('Máscara de texto')
    axes[1, 1].axis('off')

    axes[1, 2].imshow(final_rgb)
    axes[1, 2].set_title('Resultado final')
    axes[1, 2].axis('off')

    if title:
        fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


def print_debug_text_table(items, max_rows=20):
    if not items:
        print('No hay textos OCR para depurar.')
        return
    print('OCR DEBUG (OCR -> corregido -> traducido):')
    for idx, item in enumerate(items[:max_rows], start=1):
        raw_txt = item.get('raw_text', '')
        corr_txt = item.get('context_text', item.get('text', ''))
        trans_txt = item.get('translated', '')
        conf = item.get('confidence', 0.0)
        print(f'[{idx:02}] conf={conf:.2f} | ocr="{raw_txt}" | corr="{corr_txt}" | trans="{trans_txt}"')


def process_single_page(image_path, font_path=None, preview=False):
    image_path = Path(image_path)
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        raise ValueError(f'No se pudo leer la imagen: {image_path}')

    preprocessed = preprocess_image(image_bgr)
    manual_rects = []
    page_index = int(re.findall(r'\d+', image_path.stem)[0]) if re.findall(r'\d+', image_path.stem) else 0
    if ask_manual_for_page(image_path.name, page_index):
        manual_rects = select_text_regions_manual(image_bgr, image_path.name)
        if manual_rects:
            print(f'Se usaran {len(manual_rects)} regiones manuales para {image_path.name}')
        else:
            print('No se seleccionaron regiones manuales, usando OCR automatico.')

    ocr_items = detect_text(image_bgr, preprocessed, manual_rects=manual_rects)
    ocr_items = retry_low_confidence_regions(image_bgr, preprocessed, ocr_items)
    ocr_items = refine_ocr_boxes(ocr_items, preprocessed, image_bgr.shape)

    if not ocr_items:
        output_path = OUTPUT_DIR / image_path.name
        cv2.imwrite(str(output_path), image_bgr)
        return {
            'input_path': image_path,
            'output_path': output_path,
            'ocr_items': [],
            'detected_text': False,
            'ocr_engine': 'none',
            'preprocessed': preprocessed,
        }

    raw_sequence = [item['text'] for item in ocr_items]
    for idx, item in enumerate(ocr_items):
        item['raw_text'] = item['text']
        local_hint = ' | '.join(raw_sequence[max(0, idx - 1): min(len(raw_sequence), idx + 2)])
        corrected = correct_context(item['text'], context_hint=local_hint)
        item['context_text'] = corrected
        item['text'] = corrected
        item['translated'] = translate_text(corrected, CONFIG['target_lang'])

    page_font_size = None
    if bool(CONFIG.get('auto_calibrate_page', True)) and int(CONFIG.get('fixed_font_size', 0) or 0) <= 0:
        page_font_size = estimate_page_font_size(ocr_items)
        if page_font_size:
            print(f'Calibracion pagina {image_path.name}: font_size sugerido = {page_font_size}')

    cleaned_bgr, mask = inpaint_text_regions(image_bgr, ocr_items)
    final_bgr = render_translation(cleaned_bgr, ocr_items, font_path=font_path, forced_main_size=page_font_size)

    output_filename = f'{image_path.stem}_{CONFIG["target_lang"]}{image_path.suffix.lower()}'
    output_path = OUTPUT_DIR / output_filename
    cv2.imwrite(str(output_path), final_bgr)

    if preview:
        show_step_by_step(image_bgr, preprocessed, mask, ocr_items, final_bgr, title=image_path.name)
        print_debug_text_table(ocr_items)

    debug_mask_path = DEBUG_DIR / f'{image_path.stem}_mask.png'
    debug_preproc_path = DEBUG_DIR / f'{image_path.stem}_preproc.png'
    debug_edges_path = DEBUG_DIR / f'{image_path.stem}_edges.png'
    cv2.imwrite(str(debug_mask_path), mask)
    cv2.imwrite(str(debug_preproc_path), preprocessed['adaptive'])
    cv2.imwrite(str(debug_edges_path), preprocessed['edges'])

    return {
        'input_path': image_path,
        'output_path': output_path,
        'ocr_items': ocr_items,
        'detected_text': True,
        'ocr_engine': ocr_items[0].get('engine', 'unknown'),
        'preprocessed': preprocessed,
    }


def zip_output_folder(source_folder, zip_path):
    zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zip_file:
        for file_path in sorted(Path(source_folder).glob('*')):
            if file_path.is_file() and file_path.suffix.lower() in IMAGE_EXTENSIONS:
                zip_file.write(file_path, arcname=file_path.name)
    return zip_path


def run_pipeline():
    clear_folder(INPUT_DIR)
    clear_folder(OUTPUT_DIR)
    clear_folder(DEBUG_DIR)

    input_paths = prepare_input_images()
    if not input_paths:
        raise ValueError('No se encontraron imágenes de entrada válidas.')

    print(f'Archivos a procesar: {len(input_paths)}')
    print(f"Motor OCR: {CONFIG['ocr_engine']}")
    print(f"Modo seleccion manual: {CONFIG['manual_box_mode']}")
    print(f"Corrección IA activa: {CONFIG['use_ai_context_correction']}")
    print(f"Idioma de salida: {CONFIG['target_lang']} - {SUPPORTED_TARGET_LANGS.get(CONFIG['target_lang'], CONFIG['target_lang'])}")

    font_path = find_font_path()
    if font_path:
        print('Fuente detectada:', font_path)
    else:
        print('No se encontró una fuente del sistema; se usará la fuente por defecto.')

    results = []
    for index, image_path in enumerate(tqdm(input_paths, desc='Procesando páginas')):
        preview = bool(CONFIG['show_previews']) and index < int(CONFIG['preview_pages'])
        try:
            result = process_single_page(image_path, font_path=font_path, preview=preview)
            results.append(result)
            if result['detected_text']:
                print(f"[OK] {image_path.name}: {len(result['ocr_items'])} regiones detectadas")
            else:
                print(f"[INFO] {image_path.name}: sin texto detectado, se copia original")
        except Exception as exc:
            print(f"[ERROR] Fallo en {image_path.name}: {exc}")

    zip_path = zip_output_folder(OUTPUT_DIR, OUTPUT_ROOT / CONFIG['output_zip_name'])
    print('Imágenes traducidas guardadas en:', OUTPUT_DIR)
    print('ZIP listo en:', zip_path)

    if IN_COLAB and files is not None:
        try:
            files.download(str(zip_path))
        except Exception as exc:
            print('No se pudo iniciar la descarga automática:', exc)

    return results, zip_path

In [ ]:
# 6) Helper opcional: obtener coordenadas x1,y1,x2,y2
# Usa esta celda para obtener coordenadas por texto, porcentaje o clic.


def _parse_coord_line(raw):
    raw = raw.strip()
    if not raw:
        return None
    parts = [p.strip() for p in raw.split(',')]
    if len(parts) != 4:
        return None
    try:
        x1, y1, x2, y2 = [int(float(v)) for v in parts]
    except Exception:
        return None
    x1, x2 = min(x1, x2), max(x1, x2)
    y1, y2 = min(y1, y2), max(y1, y2)
    if (x2 - x1) < 5 or (y2 - y1) < 5:
        return None
    return (x1, y1, x2, y2)


def _show_reference_image(image_bgr, title='Vista de referencia para coordenadas'):
    shown = False

    if IN_COLAB:
        try:
            from google.colab.patches import cv2_imshow
            print(title)
            cv2_imshow(image_bgr)
            shown = True
        except Exception:
            shown = False

    if not shown:
        try:
            image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
            fig, ax = plt.subplots(1, 1, figsize=(12, 8))
            ax.imshow(image_rgb)
            ax.set_title(title)
            ax.axis('off')
            plt.show()
            shown = True
        except Exception:
            shown = False

    if not shown:
        try:
            from IPython.display import display
            from PIL import Image as PILImage
            image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
            display(PILImage.fromarray(image_rgb))
            print(title)
            shown = True
        except Exception:
            shown = False

    return shown


def _save_grid_reference(image_bgr, filename='coord_reference_grid.jpg', step=100):
    h, w = image_bgr.shape[:2]
    canvas = image_bgr.copy()
    step = max(50, int(step))

    for x in range(0, w, step):
        cv2.line(canvas, (x, 0), (x, h - 1), (0, 255, 255), 1)
        cv2.putText(canvas, str(x), (x + 3, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 255), 1, cv2.LINE_AA)

    for y in range(0, h, step):
        cv2.line(canvas, (0, y), (w - 1, y), (0, 255, 255), 1)
        cv2.putText(canvas, str(y), (5, y + 15), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 255), 1, cv2.LINE_AA)

    output_path = DEBUG_DIR / filename
    output_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(output_path), canvas)
    return output_path


def show_coordinate_reference(image_path=None, image_bgr=None, grid_step=100):
    """Muestra imagen original y una version con grilla para facilitar seleccion por pixeles."""
    if image_bgr is None:
        if not image_path:
            raise ValueError('Debes indicar image_path o image_bgr.')
        image_bgr = cv2.imread(str(image_path))
        if image_bgr is None:
            raise ValueError(f'No se pudo leer la imagen: {image_path}')

    h, w = image_bgr.shape[:2]
    print(f'Tamano de imagen: {w}x{h}')

    _show_reference_image(image_bgr, title='Imagen original')

    grid_path = _save_grid_reference(image_bgr, filename='coord_reference_grid.jpg', step=grid_step)
    grid_bgr = cv2.imread(str(grid_path))
    if grid_bgr is not None:
        _show_reference_image(grid_bgr, title=f'Grilla de coordenadas (paso={max(50, int(grid_step))} px)')

    print(f'Grilla guardada en: {grid_path}')
    return grid_path


def _get_box_from_text(image_bgr):
    h, w = image_bgr.shape[:2]
    shown = _show_reference_image(image_bgr, title='Ubica la region y escribe: x1,y1,x2,y2')

    # Siempre generamos referencia con grilla para facilitar lectura exacta de pixeles.
    grid_path = _save_grid_reference(image_bgr)
    print(f'Referencia con grilla: {grid_path}')

    if not shown:
        print('No se pudo renderizar la imagen en salida, pero la grilla fue generada.')

    print(f'Entrada manual de coordenadas. Tamano de imagen: {w}x{h}')
    print('Escribe coordenadas en formato x1,y1,x2,y2')
    while True:
        raw = input('Region: ').strip()
        rect = _parse_coord_line(raw)
        if rect is None:
            print('Formato invalido. Ejemplo: 120,80,430,210')
            continue
        x1, y1, x2, y2 = rect
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w - 1, x2), min(h - 1, y2)
        coord_line = f'{x1},{y1},{x2},{y2}'
        print('Coordenadas detectadas por texto:')
        print(coord_line)
        print('Copialas y pegalas cuando el notebook pida: Region: x1,y1,x2,y2')
        return (x1, y1, x2, y2)


def _get_box_from_percent(image_bgr):
    h, w = image_bgr.shape[:2]
    print('Modo porcentaje (sin necesidad de ver la imagen).')
    print('Formato: x1,y1,x2,y2 en porcentaje (0-100). Ejemplo: 10,20,80,55')

    while True:
        raw = input('Region %: ').strip()
        parts = [p.strip() for p in raw.split(',')]
        if len(parts) != 4:
            print('Formato invalido. Debe tener 4 valores.')
            continue
        try:
            px1, py1, px2, py2 = [float(v) for v in parts]
        except Exception:
            print('Solo numeros. Ejemplo: 10,20,80,55')
            continue

        px1, px2 = min(px1, px2), max(px1, px2)
        py1, py2 = min(py1, py2), max(py1, py2)

        if px1 < 0 or py1 < 0 or px2 > 100 or py2 > 100:
            print('Cada porcentaje debe estar entre 0 y 100.')
            continue

        x1 = int(round((px1 / 100.0) * (w - 1)))
        y1 = int(round((py1 / 100.0) * (h - 1)))
        x2 = int(round((px2 / 100.0) * (w - 1)))
        y2 = int(round((py2 / 100.0) * (h - 1)))

        rect = _parse_coord_line(f'{x1},{y1},{x2},{y2}')
        if rect is None:
            print('La region resultante es muy pequena. Intenta con un area mayor.')
            continue

        x1, y1, x2, y2 = rect
        coord_line = f'{x1},{y1},{x2},{y2}'
        print('Coordenadas convertidas desde porcentaje:')
        print(coord_line)
        print('Copialas y pegalas cuando el notebook pida: Region: x1,y1,x2,y2')
        return rect


def get_box_from_clicks(image_path=None, image_bgr=None, timeout=90, fallback_to_text=True, mode='auto'):
    """
    mode:
      - 'auto': en Colab usa texto; fuera de Colab intenta clic.
      - 'click': fuerza seleccion por clic.
      - 'text': fuerza entrada manual por pixeles.
      - 'percent': seleccion manual por porcentaje (0-100).
    """
    if image_bgr is None:
        if not image_path:
            raise ValueError('Debes indicar image_path o image_bgr.')
        image_bgr = cv2.imread(str(image_path))
        if image_bgr is None:
            raise ValueError(f'No se pudo leer la imagen: {image_path}')

    mode = str(mode).strip().lower()
    if mode not in ('auto', 'click', 'text', 'percent'):
        mode = 'auto'

    if mode == 'percent':
        return _get_box_from_percent(image_bgr)

    # En Colab, ginput suele fallar: preferimos texto por defecto.
    if mode == 'text' or (mode == 'auto' and IN_COLAB):
        return _get_box_from_text(image_bgr)

    if mode == 'click' and IN_COLAB:
        print('Aviso: en Colab el modo click puede no funcionar segun el backend.')
        if fallback_to_text:
            print('Se usara modo texto.')
            return _get_box_from_text(image_bgr)
        return None

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.imshow(image_rgb)
    ax.set_title('Haz 2 clics: esquina superior-izquierda y esquina inferior-derecha')
    ax.axis('off')
    plt.show()

    points = plt.ginput(2, timeout=timeout)
    plt.close(fig)

    if len(points) >= 2:
        x1, y1 = points[0]
        x2, y2 = points[1]
        x1, x2 = int(min(x1, x2)), int(max(x1, x2))
        y1, y2 = int(min(y1, y2)), int(max(y1, y2))
        coord_line = f'{x1},{y1},{x2},{y2}'
        print('Coordenadas detectadas por clic:')
        print(coord_line)
        print('Copialas y pegalas cuando el notebook pida: Region: x1,y1,x2,y2')
        return (x1, y1, x2, y2)

    print('No se capturaron 2 puntos por clic.')
    if not fallback_to_text:
        print('Intenta de nuevo o activa fallback_to_text=True.')
        return None

    return _get_box_from_text(image_bgr)


# Ejemplos de uso:
# sample_image = INPUT_DIR / '3.jpg'
# show_coordinate_reference(image_path=sample_image, grid_step=80)
# get_box_from_clicks(image_path=sample_image, mode='auto')
# get_box_from_clicks(image_path=sample_image, mode='text')
# get_box_from_clicks(image_path=sample_image, mode='percent')

In [ ]:
# 7) Configuración rápida y ejecución
# Ejecuta esta celda para iniciar la carga y procesamiento en Colab.

CONFIG['source_lang'] = 'en'
CONFIG['target_lang'] = 'es'
CONFIG['ocr_engine'] = 'easyocr'        # 'easyocr' | 'tesseract' | 'paddle'
CONFIG['ocr_min_confidence'] = 0.40
CONFIG['ocr_use_paragraph_pass'] = True
CONFIG['ocr_include_gray_pass'] = True
CONFIG['ocr_scales'] = [1.0, 1.35]
CONFIG['enable_ocr_correction'] = True
CONFIG['use_ai_context_correction'] = False
CONFIG['gemini_api_key'] = ''           # Si activas IA, pega aquí tu API key
CONFIG['gemini_model'] = 'gemini-1.5-flash'
CONFIG['keep_original_subtitle'] = False
CONFIG['use_gpu'] = False
CONFIG['show_previews'] = True
CONFIG['preview_pages'] = 1
CONFIG['retry_low_confidence'] = True
CONFIG['manual_box_mode'] = 'ask'    # 'ask' | 'always' | 'never'
CONFIG['manual_selection_method'] = 'auto'  # 'auto' | 'text' | 'click'
CONFIG['use_url_input'] = False
CONFIG['input_url'] = ''

# Si quieres probar otros idiomas de salida:
# CONFIG['target_lang'] = 'fr'
# CONFIG['target_lang'] = 'pt'
# CONFIG['target_lang'] = 'it'

results, zip_path = run_pipeline()
print('Páginas procesadas:', len(results))
results[:1] if results else []